# Fixed Oscilloscope Analysis
This notebook has been automatically repaired to properly load 640-point segments, filter the data, plot the signals, and safely handle curve fitting.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import simpson as simps
from scipy.optimize import curve_fit
import warnings
warnings.filterwarnings('ignore')

In [ ]:
def csv_reader(file_name, save=False):
    data_csv = np.genfromtxt(file_name, delimiter=",", skip_header=3)
    points_per_segment = 640
    size = len(data_csv) // points_per_segment
    print(f"Reading {file_name} - Events: {size}")
    
    time_csv, ch1_csv, ch2_csv = data_csv[:, 0], data_csv[:, 1], data_csv[:, 2]
    
    common_dtype = [
        ('event_number', np.int32), ('start_time', np.float64),
        ('1_peak_location', int), ('1_peak_value', np.float32),
        ('2_peak_location', int), ('2_peak_value', np.float32),
    ]
    if save:
        common_dtype.append(('data', np.float32, points_per_segment))
        
    nd_array = np.full(size, np.nan, dtype=common_dtype)
    channels_data = {1: nd_array.copy(), 2: nd_array.copy()}
    
    print("Processing data (takes ~1 second)...")
    for i in range(size):
        start, end = i * points_per_segment, (i + 1) * points_per_segment
        time_i, ch1_i, ch2_i = time_csv[start:end], ch1_csv[start:end]*(-1), ch2_csv[start:end]*(-1)
        
        if save:
            channels_data[1]['data'][i], channels_data[2]['data'][i] = ch1_i, ch2_i
            
        for ch, ch_i in [(1, ch1_i), (2, ch2_i)]:
            channels_data[ch]['event_number'][i] = i
            channels_data[ch]['start_time'][i] = time_i[0]
            
            p1_loc = int(np.argmax(ch_i))
            channels_data[ch]['1_peak_value'][i] = ch_i[p1_loc]
            channels_data[ch]['1_peak_location'][i] = p1_loc
            
            # Find 2nd peak (search after 1st peak + 33)
            p1_r = min(points_per_segment - 1, p1_loc + 33)
            if p1_r < points_per_segment - 1:
                p2_loc = int(np.argmax(ch_i[p1_r:]) + p1_r)
                channels_data[ch]['2_peak_value'][i] = ch_i[p2_loc]
                channels_data[ch]['2_peak_location'][i] = p2_loc
            else:
                channels_data[ch]['2_peak_value'][i] = 0
                channels_data[ch]['2_peak_location'][i] = points_per_segment - 1
                
    return channels_data

In [ ]:
# Load measurement 0
data = csv_reader("/Users/drorta/alpha-1/measurements/nd-2-measurement-0.csv", save=True)
ch_1, ch_2 = data[1], data[2]

In [ ]:
# Filter out bad events
valid_events = (
    (np.abs(ch_1['1_peak_location'] - ch_2['1_peak_location']) < 10) &  # Peaks align in time
    (ch_1['1_peak_value'] > 0.005) &                                    # Ch1 has signal
    (ch_2['1_peak_value'] > 0.005)                                      # Ch2 has signal
)

ch_1_filtered = ch_1[valid_events]
ch_2_filtered = ch_2[valid_events]

print(f"Events after filtering: {len(ch_2_filtered)}")

In [ ]:
# Plot the first 5 valid signals
for i in range(min(5, len(ch_2_filtered))): 
    plt.figure(figsize=(10, 6))
    
    plt.subplot(2, 1, 1)
    plt.plot(ch_1_filtered['data'][i])
    plt.axvline(ch_1_filtered['1_peak_location'][i], c='g', linestyle='--', label='Peak 1')
    plt.axvline(ch_1_filtered['2_peak_location'][i], c='r', linestyle='--', label='Peak 2')
    plt.title(f"Event {i} - Channel 1")
    plt.legend()
    
    plt.subplot(2, 1, 2)
    plt.plot(ch_2_filtered['data'][i])
    plt.axvline(ch_2_filtered['1_peak_location'][i], c='g', linestyle='--', label='Peak 1')
    plt.axvline(ch_2_filtered['2_peak_location'][i], c='r', linestyle='--', label='Peak 2')
    plt.title(f"Event {i} - Channel 2")
    
    plt.tight_layout()
    plt.show()

In [ ]:
def peaks_time_diff(ch2):
    time_diff = ch2['2_peak_location'] - ch2['1_peak_location']
    return time_diff * 2.5e-9 * (2000/781)

def expo(t, A, tau_fit):
    return A * np.exp(-t / tau_fit)

In [ ]:
# Curve Fitting
data_times = peaks_time_diff(ch_2_filtered)

plt.figure(figsize=(8, 5))
a, b, _ = plt.hist(data_times, bins=np.arange(0.15e-6, 5.05e-6, 1e-6))
x_data = (b[:-1] + b[1:]) / 2

if np.count_nonzero(a) > 1:
    try:
        popt, _ = curve_fit(expo, x_data, a, p0=(max(a), 2.2e-6), maxfev=5000)
        t_plot = np.arange(0, 5.1e-6, 0.1e-6)
        plt.plot(t_plot, expo(t_plot, *popt), 'r--', label=f'Fit: τ = {popt[1]*1e6:.2f} μs')
        plt.legend()
    except RuntimeError:
        print("Curve fitting failed to converge.")
else:
    print("Not enough spread in data to fit an exponential curve (all events fell in 1 bin).")

plt.title("Muon Decay Times")
plt.xlabel("Time (s)")
plt.ylabel("Counts")
plt.show()